# 01 - Clean, join, build ball-level state features
Nothing is dropped here. Super overs, associate nations and A-team games are kept and flagged.
Output: outputs/balls_features.parquet

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA, OUT = ROOT / 'data', ROOT / 'outputs'
OUT.mkdir(exist_ok=True)

FULL_MEMBERS = {'India', 'Pakistan', 'West Indies', 'New Zealand', 'Bangladesh', 'Sri Lanka', 'England',
                'Afghanistan', 'Australia', 'Zimbabwe', 'South Africa', 'Ireland'}

## Load and build match-level table

In [2]:
bbb = pd.read_csv(DATA / 'ball_by_ball.csv')
m = pd.read_csv(DATA / 'matches.csv', parse_dates=['match_date'])
comp = pd.read_csv(DATA / 'competitions.csv')

m = m.merge(comp, on='comp_id', how='left')
m['comp_group'] = np.select([m.comp_name.str.startswith('IPL'), m.match_format.eq('T20I')], ['IPL', 'T20I'], 'A_TEAM')
m['status'] = np.select([m.result.str.contains('No Result|Abandon', case=False),
                         m.result.str.contains('Tie', case=False)], ['no_result', 'tied'], 'ok')
m['dls'] = m.revised_target.notna()

teams = pd.concat([
    m[['home_team_id', 'home_team']].set_axis(['team_id', 'team'], axis=1),
    m[['away_team_id', 'away_team']].set_axis(['team_id', 'team'], axis=1),
]).drop_duplicates('team_id').set_index('team_id').team

KEEP = ['match_id', 'comp_id', 'comp_name', 'season', 'comp_group', 'match_date', 'stage', 'knock_out',
        'venue_country', 'win_team_id', 'status', 'dls', 'first_inn_revised_overs',
        'second_inn_revised_overs', 'revised_target']
b = bbb.merge(m[KEEP], on='match_id', how='left', validate='m:1')
assert b.comp_group.notna().all()
b = b.sort_values(['match_id', 'innings', 'innings_ball_num']).reset_index(drop=True)

## Tiers (kept as features, used later for opposition adjustment)

In [3]:
b['bat_team'] = b.batting_team_id.map(teams)
b['bowl_team'] = b.bowling_team_id.map(teams)


def tier(names, g):
    return np.select([g.eq('IPL'), g.eq('A_TEAM'), names.isin(FULL_MEMBERS)], ['IPL', 'A', 'FM'], 'ASSOC')


b['bat_tier'] = tier(b.bat_team, b.comp_group)
b['bowl_tier'] = tier(b.bowl_team, b.comp_group)

## Delivery flags
`runs` = bat runs only. Extras live in `extras1_runs`.

In [4]:
ILLEGAL = ['wide', 'Overstep', 'other']
b['is_super_over'] = b.innings > 2
b['legal'] = ~b.extras_type.isin(ILLEGAL)
b['faced'] = b.extras_type.ne('wide')
b['tot'] = b.runs + b.extras1_runs
b['bowler_runs'] = b.runs + np.where(b.extras_type.isin(ILLEGAL), b.extras1_runs, 0)
b['bowler_wkt'] = b.is_wicket & b.dismissal_type.isin(['Caught', 'Bowled', 'LBW', 'Stump', 'Hit Wicket'])
b['batter_out'] = b.is_wicket & b.dismissal_type.ne('Retire Out') & b.dismissed_player_id.eq(b.striker_id)

## State before each delivery

In [5]:
b['lg'] = b.legal.astype(int)
b['w'] = b.is_wicket.astype(int)
b['fc'] = b.faced.astype(int)
key = ['match_id', 'innings']

g = b.groupby(key, sort=False)
b['score'] = g.tot.cumsum() - b.tot
b['wkts'] = g.w.cumsum() - b.w
b['lb'] = g.lg.cumsum() - b.lg
b['max_balls'] = np.select([b.is_super_over, b.innings.eq(1)], [6, b.first_inn_revised_overs.fillna(120)],
                           b.second_inn_revised_overs.fillna(120))
b['balls_left'] = b.max_balls - b.lb

tot_inn = b.groupby(key).tot.sum().rename('prev_total').reset_index()
tot_inn['innings'] += 1
b = b.merge(tot_inn, on=key, how='left')
b['target'] = np.where(b.innings.eq(2) & b.revised_target.notna(), b.revised_target, b.prev_total + 1)
b.loc[b.innings.isin([1, 3]), 'target'] = np.nan
b['runs_needed'] = b.target - b.score
b['rrr'] = np.where(b.balls_left > 0, b.runs_needed / b.balls_left * 6, np.nan)
b['crr'] = np.where(b.lb > 0, b.score / b.lb * 6, 0.0)
b['phase'] = np.select([b.is_super_over, b.overid <= 6, b.overid <= 15], ['SO', 'PP', 'MID'], 'DEATH')

gp = b.groupby(key + ['wkts'], sort=False)
b['p_runs'] = gp.tot.cumsum() - b.tot
b['p_balls'] = gp.lg.cumsum() - b.lg

gb = b.groupby(key + ['striker_id'], sort=False)
b['bat_runs_before'] = gb.runs.cumsum() - b.runs
b['bat_balls_before'] = gb.fc.cumsum() - b.fc

## Pressure build-up features
dot streaks, dots in the last 6 balls, rising required rate, free hits, bowler over momentum

In [6]:
b['bdot'] = b.faced & b.runs.eq(0)                          # batter dot ball (byes / leg-byes count as dots)
b['is_bnd'] = b.is_four | b.is_six
b['bowl_dot'] = b.legal & b.bowler_runs.eq(0)               # bowler dot: legal ball, nothing conceded
b['bowl_extra'] = ~b.legal                                  # wide / no-ball / other
b['free_hit'] = b.groupby(key).extras_type.shift(1).eq('Overstep').astype(int)

k3 = ['match_id', 'innings', 'striker_id']
f = b.loc[b.faced, k3 + ['innings_ball_num', 'bdot']].copy()
f['d'] = f.bdot.astype(int)
f['nd'] = 1 - f.d
f['grp'] = f.groupby(k3).nd.cumsum()
f['streak_incl'] = f.groupby(k3 + ['grp']).d.cumsum()
f['dot_streak'] = f.groupby(k3).streak_incl.shift(1).fillna(0)          # dots in a row just before this ball
f['cs'] = f.groupby(k3).d.cumsum()
f['dots_last6'] = f.groupby(k3).cs.shift(1).fillna(0) - f.groupby(k3).cs.shift(7).fillna(0)
b = b.merge(f[['match_id', 'innings', 'innings_ball_num', 'dot_streak', 'dots_last6']],
            on=['match_id', 'innings', 'innings_ball_num'], how='left')
b[['dot_streak', 'dots_last6']] = b[['dot_streak', 'dots_last6']].fillna(0)

slot = b.groupby(key + ['lb']).rrr.first().reset_index().rename(columns={'lb': 'lb6', 'rrr': 'rrr_6ago'})
b['lb6'] = (b.lb - 6).clip(lower=0)
b = b.merge(slot, on=key + ['lb6'], how='left')
b['rrr_trend'] = b.rrr - b.rrr_6ago                                      # > 0: required rate has been rising
b = b.drop(columns=['lb6', 'rrr_6ago'])

ov = ['match_id', 'innings', 'overid']
b['_bd'], b['_be'] = b.bowl_dot.astype(int), b.bowl_extra.astype(int)
b['bowler_dots_over'] = b.groupby(ov)._bd.cumsum() - b._bd              # dots already bowled in this over
b['bowler_extras_over'] = b.groupby(ov)._be.cumsum() - b._be
b = b.drop(columns=['_bd', '_be'])

## Label for the win-probability model

In [7]:
b['bat_win'] = np.where(b.win_team_id.isna(), np.nan, b.win_team_id.eq(b.batting_team_id).astype(float))
b['train_ok'] = b.innings.isin([1, 2]) & b.status.eq('ok') & ~b.dls & b.bat_win.notna()
b = b.drop(columns=['lg', 'w', 'fc'])

## Checks

In [8]:
print('rows', len(b), '| matches', b.match_id.nunique(), '| players batting', b.striker_id.nunique(),
      '| bowling', b.bowler_id.nunique())
print('\nballs by competition group x batting tier')
print(b.groupby(['comp_group', 'bat_tier']).agg(balls=('tot', 'size'), matches=('match_id', 'nunique')))
print('\nsuper-over balls:', int(b.is_super_over.sum()), 'in', b[b.is_super_over].match_id.nunique(), 'matches')
mm = b.drop_duplicates('match_id')
print('status:', mm.status.value_counts().to_dict(), '| DLS matches:', int(mm.dls.sum()))
print('train rows:', int(b.train_ok.sum()), 'in', b[b.train_ok].match_id.nunique(), 'matches')

reg = b[b.innings.isin([1, 2])]
print('balls_left min (regulation):', reg.balls_left.min(), '| innings-2 rows without target:',
      int(reg[reg.innings.eq(2)].target.isna().sum()))

# label vs ball-data agreement: did innings-2 total reach the target?
t2 = reg[reg.innings.eq(2) & reg.train_ok].groupby('match_id').agg(t=('tot', 'sum'), tg=('target', 'first'),
                                                                   win=('bat_win', 'first'))
print('label agrees with ball-data chase result: %.1f%% of %d matches' % (((t2.t >= t2.tg) == (t2.win == 1)).mean() * 100, len(t2)))

reg2 = b[b.innings.isin([1, 2]) & b.legal]
fh = reg2.groupby('free_hit').agg(balls=('tot', 'size'), runs_per_ball=('tot', 'mean'), wkt_rate=('is_wicket', 'mean'))
print('\nfree-hit effect on the next legal ball:'); print(fh.round(3))
print('dot_streak mean/max:', round(b[b.faced].dot_streak.mean(), 2), int(b[b.faced].dot_streak.max()), '| rrr_trend > 0 share in chases:',
      round((b[b.innings.eq(2)].rrr_trend > 0).mean(), 3))
b.to_parquet(OUT / 'balls_features.parquet')
print('saved', OUT / 'balls_features.parquet')

rows 200999 | matches 867 | players batting 888 | bowling 647

balls by competition group x batting tier
                     balls  matches
comp_group bat_tier                
A_TEAM     A          6985       30
IPL        IPL       87687      367
T20I       ASSOC      7697       52
           FM        98630      455

super-over balls: 38 in 6 matches
status: {'ok': 838, 'no_result': 24, 'tied': 5} | DLS matches: 23
train rows: 194393 in 816 matches
balls_left min (regulation): 1.0 | innings-2 rows without target: 0
label agrees with ball-data chase result: 99.3% of 816 matches

free-hit effect on the next legal ball:
           balls  runs_per_ball  wkt_rate
free_hit                                 
0         191895          1.416     0.056
1            517          2.371     0.004
dot_streak mean/max: 0.49 11 | rrr_trend > 0 share in chases: 0.602
saved d:\Projects\superr\outputs\balls_features.parquet
